In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
import os
 
# Path to your Who&When JSON file — edit this.
DATASET_DIR = "./ww"
SUBSET = "hand-crafted"
 
# Model names used in the evaluation plan.
MODEL_LLAMA = "/data/hoang/resources/models/meta-llama/Llama-3.1-8B-Instruct"
MODEL_QWEN  = "/data/hoang/resources/models/Qwen/Qwen3-4B" 

# Which model to load in this session.
MODEL_NAME = MODEL_QWEN  
 
DEVICE = "6"   # "cpu" for a CPU-only sanity pass (slow)

In [2]:
from gradnorm.data import load_dataset, build_context, select_context
from utils.graph import get_dependency_dict, derive_llm_inputs

from transformers import PreTrainedModel, PreTrainedTokenizer
from torch import Tensor
from gradnorm.gradnorm import _ntp_loss


trajectories  = load_dataset(DATASET_DIR, subset=SUBSET)
traj = trajectories[11]
len(traj.history)
deps = get_dependency_dict(derive_llm_inputs(traj.history))
# traj.history

In [3]:
deps

{0: [],
 1: [0],
 2: [0, 1],
 3: [2],
 4: [0, 1, 3],
 5: [0, 1, 3, 4],
 6: [5],
 7: [5],
 8: [0, 1, 3, 4, 6],
 9: [0, 1, 3, 4, 6, 8],
 10: [9],
 11: [9],
 12: [0, 1, 3, 4, 6, 8, 10],
 13: [0, 1, 3, 4, 6, 8, 10, 12],
 14: [13],
 15: [13],
 16: [0, 1, 3, 4, 6, 8, 10, 12, 14],
 17: [0, 1, 3, 4, 6, 8, 10, 12, 14, 16],
 18: [0, 1, 3, 4, 6, 8, 10, 12, 14, 16],
 19: [0, 1, 3, 4, 6, 8, 10, 12, 14, 16]}

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"\nLoading tokeniser: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model ({MODEL_NAME}) → {DEVICE}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype  = torch.bfloat16,
    device_map   = {"": int(DEVICE)},
)
model.eval()
n_params = sum(p.numel() for p in model.parameters())

print(f"  {n_params / 1e9:.2f}B parameters loaded.\n")


Loading tokeniser: /data/hoang/resources/models/Qwen/Qwen3-4B


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model (/data/hoang/resources/models/Qwen/Qwen3-4B) → 6


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  4.02B parameters loaded.



In [5]:
def pad_encoded(encoded: dict[Tensor, int]):
    padded = tokenizer.pad(
        [{"input_ids": ids} for ids in encoded["input_ids"]],
        return_tensors="pt",
        padding_side="left",
        padding="max_length",
        max_length=4096
    )
    padded_ids, attention_mask = padded["input_ids"], padded["attention_mask"]
    num_padded = int(attention_mask.shape[1] - attention_mask.sum(dim=1))
    padded_ctx_len = encoded["ctx_len"] + num_padded
    return {
        "input_ids": padded_ids, 
        "attention_mask": attention_mask, 
        "ctx_len": padded_ctx_len
    }

In [ ]:
# for p in model.parameters():
#     if p.grad is not None:
#         p.grad = None
#     # p.requires_grad_(True)

False

In [ ]:
# # ── Step selection ──────────────────────────────────────────────
# step_idx = 16
# context_builder = build_context
# role = traj.history[step_idx].get("role", f"step_{step_idx}")

# # ── Tokenise ────────────────────────────────────────────────────
# encoded   = context_builder(
#     traj.history, 
#     step_idx, 
#     tokenizer,
#     max_tokens=4096
# )
# # encoded = pad_encoded(encoded)

# attention_mask = encoded.get("attention_mask", None)

# input_ids = encoded["input_ids"].to(f"cuda:{DEVICE}")   # (1, seq_len)
# ctx_len   = encoded["ctx_len"]
# seq_len   = input_ids.shape[1]

# print(f"seq_len: {seq_len}, ctx_len: {ctx_len}")

seq_len: 3221, ctx_len: 2890


In [ ]:
def gradnorm_standard(
    model:          PreTrainedModel,
    input_ids:      Tensor,
    attention_mask: Tensor,
    ctx_len:        int,
    normalize:      bool = True,
) -> float:
    # ── Compute gradients ────────────────────────────────────────────
    logits = model(
        input_ids, 
        attention_mask, 
        use_cache=False
    ).logits  # (1, seq_len, vocab)
    loss   = _ntp_loss(logits, input_ids, ctx_len)
    loss.backward()

    # ── Compute score ────────────────────────────────────────────────
    target_weights = [layer for layer in model.model.layers]
    target_weights += [model.lm_head]

    n_layers = len(model.model.layers)
    module_names = [f"layer_{i}" for i in range(n_layers)] + ["lm_head"]

    statistics = {}
    for name, module in zip(module_names, target_weights):
        n_params = sum(p.numel() for p in module.parameters())
        l1_norm = sum(p.grad.detach().abs().sum().item() for p in module.parameters()) 
        l2_norm = sum(p.grad.detach().norm(2).sum().item() for p in module.parameters())
        if normalize:
            l1_norm = l1_norm / n_params
            l2_norm = l2_norm / n_params

        statistics[name] = {
            "l1_norm": l1_norm,
            "l2_norm": l2_norm,
        }

    # ── Cleanup: zero grads, re-freeze ───────────────────────────────
    for p in model.parameters():
        if p.grad is not None:
            p.grad = None
        # p.requires_grad_(False)

    return statistics

In [7]:

trajectories  = load_dataset(DATASET_DIR, subset=SUBSET)
traj = trajectories[11]
context_builder = build_context
MAX_TOKENS = 4096

tests = []

for step_idx in range(len(traj.history)):
    # ── Tokenise ────────────────────────────────────────────────────
    encoded   = context_builder(
        traj.history, 
        step_idx, 
        tokenizer,
        max_tokens=MAX_TOKENS
    )
    encoded = pad_encoded(encoded)

    input_ids = encoded["input_ids"].to(f"cuda:{DEVICE}")   # (1, seq_len)
    attention_mask = encoded.get("attention_mask", None)
    ctx_len   = encoded["ctx_len"]
    seq_len   = input_ids.shape[1]

    print(f"seq_len: {seq_len}, ctx_len: {ctx_len}")

    statistics = gradnorm_standard(
        model, input_ids, attention_mask, ctx_len
    )
    out = {
        "metadata": {
            "filename": traj.filename,
            "step_idx": step_idx,
            "input_ids": input_ids.tolist(),
            "attention_mask": attention_mask.tolist(),
            "ctx_len": ctx_len,
        },
        "statistics": statistics
    }
    tests.append(out)

seq_len: 4096, ctx_len: 4028
seq_len: 4096, ctx_len: 3366


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.32 GiB. GPU 6 has a total capacity of 44.40 GiB of which 1.23 GiB is free. Including non-PyTorch memory, this process has 43.17 GiB memory in use. Of the allocated memory 39.58 GiB is allocated by PyTorch, and 2.49 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
statistics = gradnorm_standard(
    model, input_ids, attention_mask, ctx_len
)
os.makedirs("tests", exist_ok=True)
with open("tests/standard.json", "w") as f:
    out = {
        "metadata": {
            "trajectory": None,
            "step_idx": None,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "ctx_len": ctx_len,
        },
        "statistics": statistics
    }
    json.dump(statistics, f, indent=2)

In [37]:
logits = model(
    input_ids, 
    attention_mask, 
    use_cache=False
).logits  # (1, seq_len, vocab)
loss   = _ntp_loss(logits, input_ids, ctx_len)
loss.backward()

In [38]:
import os
import json

target_weights = [layer for layer in model.model.layers]
target_weights += [model.lm_head]

n_layers = len(model.model.layers)
module_names = [f"layer_{i}" for i in range(n_layers)] + ["lm_head"]

statistics = {}
for name, module in zip(module_names, target_weights):
    n_params = sum(p.numel() for p in module.parameters())
    # print(f"{name}: {n_params} params")
    l1_norm = sum(p.detach().abs().sum().item() for p in module.parameters()) 
    l2_norm = sum(p.detach().norm(2).sum().item() for p in module.parameters())
    statistics[name] = {
        "l1_norm": l1_norm / n_params,
        "l2_norm": l2_norm / n_params,
    }

In [39]:
os.makedirs("tests", exist_ok=True)
with open("tests/standard-gradnorm-statistics-unpadded.json", "w") as f:
    json.dump(statistics, f, indent=2)

In [ ]:
# ── Clean up memory ─────────────────────────────────────────────
def memory_accounting():
    device = f"cuda:{DEVICE}"
    torch.cuda.empty_cache()
    alloc_mb = torch.cuda.memory_allocated(device) / 1e6
    reserv_mb = torch.cuda.memory_reserved(device) / 1e6
    print(f"[{device}] CUDA allocated : {alloc_mb:.1f} MB")
    print(f"[{device}] CUDA reserved  : {reserv_mb:.1f} MB")


for p in model.parameters():
    if p.grad is not None:
        p.grad = None
        
memory_accounting()

[cuda:6] CUDA allocated : 9307.7 MB
[cuda:6] CUDA reserved  : 9353.3 MB


325